In [1]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")

import jax
jax.config.update("jax_enable_x64", True)

from pyscf import gto, scf, cc
import os
import numpy as np

atoms = '''
C 0.0 0.0 0.0
N 0.0 0.0 1.16739
'''

mol = gto.M(
        atom=atoms,
        basis="ccpvdz",
        spin=1,
        max_memory=40000,
        verbose=4
        )
mol.build()


mf = scf.UHF(mol)
mf.kernel()

stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break


mycc = cc.CCSD(mf)
mycc.set_frozen()
mycc.kernel()

eccs = mycc.energy(mycc.t1,(mycc.t2[0]*0,mycc.t2[1]*0,mycc.t2[2]*0))
print(f"ccs energy = {mf.e_tot+eccs}")

System: uname_result(system='Linux', node='sharmagroup-rn', release='7.0.0-28-generic', version='#28~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Wed Jul  1 15:50:57 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Tue Jul 21 22:12:53 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 13
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 1
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                

In [19]:
options =  {'eql_time': 5,
            'n_blocks': 0,
            'n_walkers': 400,
            'max_memory': 8000,
            'seed': 17,
            'trial': 'upt2ccsd_wrong',
            'mix_precision': False,
            }

from afqmc import integral, launch_afqmc
integral.prep_integral(mycc, chol_cut=1e-6)


Preparing AFQMC calculation
Calculating Cholesky integrals
Alpha Cholesky shape: (207, 26, 26) 
 Beta Cholesky shape: (207, 26, 26) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [5 4]
Number of basis functions:  26
Number of Cholesky vectors: 207


In [31]:
from jax import random
import jax.numpy as jnp
from afqmc import walker_tools

def init_delta_prop_data(
    wave,
    wave_data,
    ham_data,
    options
    ):

    print("\nInitalize QMC walkers by HF")
    prop_data = {}
    prop_data["n_killed_walkers"] = 0
    prop_data["key"] = random.PRNGKey(options["seed"])

    weights0 = jnp.ones(options["n_walkers"], dtype=jnp.float64)
    norba, nocca = wave_data["mo_coeff"][0].shape
    norbb, noccb = wave_data["mo_coeff"][1].shape
    walker0 = (jnp.eye(norba)[:,:nocca], jnp.eye(norbb)[:,:noccb])
    walkers0 = walker_tools.replicate_walker(walker0, options["n_walkers"])
    overlaps0 = wave.calc_overlap(walkers0, wave_data)
    energies0 = wave.calc_energy(walkers0, ham_data, wave_data)
    energy0 = jnp.sum(overlaps0 * energies0) / jnp.sum(overlaps0)

    prop_data["walkers"] = walkers0
    prop_data["weights"] = weights0
    prop_data["overlaps"] = overlaps0
    prop_data["e_estimate"] = jnp.real(energy0)
    prop_data["pop_control_ene_shift"] = prop_data["e_estimate"]

    return prop_data

In [32]:
import time

import numpy as np

from afqmc import config, prep, sampling

from functools import partial

print = partial(print, flush=True)
init_time = time.time()

# prep.print_start()
config.setup_jax()

ham_data, ham, prop, trial, wave_data, sampler, options = prep.init_afqmc(options=options)

if "rdm1" not in wave_data:
    wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
h0 = ham_data['h0']

prop_data = init_delta_prop_data(trial, wave_data, ham_data, options)

init_e = prop_data["e_estimate"]
init_w = np.sum(prop_data["weights"])

print(init_e)


Hostname:     sharmagroup-rn
System:       Linux
Node:         sharmagroup-rn
Release:      7.0.0-28-generic
Machine:      x86_64
Processor:    x86_64
JAX backend:  GPU
JAX devices:  [CudaDevice(id=0)]
Device kind:  NVIDIA GeForce RTX 5060 Ti
Platform:     gpu

QMC Parameters
eql_time        -          5
n_blocks        -          0
n_walkers       -        400
max_memory      -       8000
seed            -         17
trial           - upt2ccsd_wrong
mix_precision   -      False
dt              -      0.005
n_exp_terms     -          6
n_prop_steps    -         50
walker_type     -        uhf
n_batch         -          1
max_error       -        0.0
nchol_chunk     -        100
free_projection -      False

Load system from Integral File
Maximum memory per walker:            20.00 MB
Maximum number of Cholesky per chunk: 969
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         207
Number of padding Cholesky:           0

QMC System
Number of electrons: (5, 4)
S

In [14]:
print(init_e)
print(mf.e_tot)

-92.21299914620258
-92.21299807704744


In [26]:
print(wave_data["mo_ta"]-wave_data["mo_coeff"][0])

[[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]]


In [28]:
print(wave_data["mo_tb"]-wave_data["mo_coeff"][1])

[[0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j]]


In [29]:
trial._calc_energy(wave_data["mo_ta"], wave_data["mo_tb"], ham_data, wave_data)

Array(-92.19388937+0.j, dtype=complex128)

In [30]:
ket

(Array([[ 1.00000000e+00+0.j,  0.00000000e+00+0.j,  0.00000000e+00+0.j,
          0.00000000e+00+0.j,  0.00000000e+00+0.j],
        [ 0.00000000e+00+0.j,  1.00000000e+00+0.j,  0.00000000e+00+0.j,
          0.00000000e+00+0.j,  0.00000000e+00+0.j],
        [ 0.00000000e+00+0.j,  0.00000000e+00+0.j,  1.00000000e+00+0.j,
          0.00000000e+00+0.j,  0.00000000e+00+0.j],
        [ 0.00000000e+00+0.j,  0.00000000e+00+0.j,  0.00000000e+00+0.j,
          1.00000000e+00+0.j,  0.00000000e+00+0.j],
        [ 0.00000000e+00+0.j,  0.00000000e+00+0.j,  0.00000000e+00+0.j,
          0.00000000e+00+0.j,  1.00000000e+00+0.j],
        [-1.59068220e-16+0.j,  4.30220723e-17+0.j,  5.03641848e-16+0.j,
         -4.58296415e-03+0.j,  1.49160753e-01+0.j],
        [-1.30012371e-18+0.j,  2.11648109e-16+0.j,  1.34385286e-16+0.j,
          1.49160753e-01+0.j,  4.58296415e-03+0.j],
        [ 5.17327322e-03+0.j, -1.32211407e-03+0.j,  1.74685336e-04+0.j,
          5.34714222e-15+0.j,  1.12919752e-15+0.j],
        

In [23]:
from afqmc import slater_tools
bra = (wave_data["mo_ta"], wave_data["mo_tb"])
ket = (prop_data["walkers"][0][0], prop_data["walkers"][1][0])
h0 = ham_data["h0"]
h1 = ham_data["h1"]
norba, norbb = bra[0].shape[0], bra[1].shape[0]
chol = (ham_data["chol"][0].reshape(-1, norba, norba),
        ham_data["chol"][1].reshape(-1, norbb, norbb))
slater_tools.u_energy(bra, ket, h0, h1, chol)

Array(-92.19388937+0.j, dtype=complex128)